# Processamento do OSRM Brasil via Docker (Alta Performance)

Este notebook automatiza o download e a compilação da malha rodoviária do Brasil inteiro utilizando o Docker dentro do Colab Pro (aproveitando os 50 GB de RAM), enviando o resultado compactado diretamente para o seu Google Drive de forma segura.

In [ ]:
%%bash
sudo apt-get update -qq
sudo apt-get install -y build-essential git cmake libboost-all-dev libtbb-dev liblua5.2-dev libxml2-dev libosmpbf-dev libprotobuf-dev protobuf-compiler pkg-config

Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
git is already the newest version (1:2.34.1-1ubuntu1.17).
protobuf-compiler is already the newest version (3.12.4-1ubuntu7.22.04.6).
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3
Use 'sudo apt autoremove' to remove them.
The following additional packages will be installed:
  javascript-common libboost-atomic-dev libboost-atomic1.74-dev
  libboost-atomic1.74.0 libboost-chrono-dev libboost-chrono1.74-dev
  libboost-chrono1.74.0 libboost-container-dev libboost-container1.74-dev
  libboost-container1.74.0 libboost-context-dev libboost-context1.74-dev
  libboost-context1.74.0 libboost-coroutine-dev libboost-coroutine1.74-dev
  libboost-coroutine1.74.0 libboost-date-time-dev libboost-date-time1.74-dev
  libboost-date-time1.74.0 lib

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 121.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
PASTA_DESTINO_DRIVE = "/content/drive/MyDrive/OSRM_Brasil_Compilado"
os.makedirs(PASTA_DESTINO_DRIVE, exist_ok=True)

print(f"✅ Google Drive conectado com sucesso!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive conectado com sucesso!


In [ ]:
%%bash
cd /content
rm -rf osrm-backend # Limpa tentativas anteriores
git clone --depth 1 --branch v5.27.1 https://github.com/Project-OSRM/osrm-backend.git
cd osrm-backend
mkdir build
cd build
cmake .. -DCMAKE_BUILD_TYPE=Release
make -j$(nproc)

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Building on a 64 bit system
-- Using GNU gold as linker.
-- Configuring OSRM in release mode
-- Configuring release mode optimizations
-- Performing Test LTO_AVAILABLE
-- Performing Test LTO_AVAILABLE - Success
-- Performing Test SUPPORTS_CXXFLAG_all
-- Performing Test SUPPORTS_CXXFLAG_all - Success
-- Performing Test SUPPORTS_CFLAG_all
-- Performing Test SUPPORTS_CFLAG_all - Success
-- Performing Test SUPPORTS_CXXFLAG_extra
-- Performing Test SUPPORTS_CXXFLAG_extra - Success
-- Performi

Cloning into 'osrm-backend'...
Note: switching to '4f3ee609ec1af40eb1f445c6706cfa5beb04c990'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


CMake Deprecation W

In [ ]:
import os

caminho_build = "/content/osrm-backend/build"
if os.path.exists(caminho_build):
    print(f"📁 Arquivos gerados dentro da pasta build:")
    for arquivo in os.listdir(caminho_build):
        if 'osrm' in arquivo:
            print(f"  - {arquivo}")
else:
    print("❌ A pasta build não foi encontrada. A compilação da Célula 2 falhou.")

📁 Arquivos gerados dentro da pasta build:
  - libosrm_partition.a
  - osrm-components
  - osrm-routed
  - osrm-datastore
  - osrm-partition
  - libosrm_contract.a
  - libosrm_guidance.a
  - libosrm_customize.a
  - libosrm_update.a
  - libosrm.pc
  - osrm-customize
  - libosrm_extract.a
  - libosrm.a
  - osrm-contract
  - osrm-extract
  - libosrm_store.a


In [ ]:
%%bash
echo "⚙️ Etapa A: Extração do Grafo (osrm-extract)..."
/content/osrm-backend/build/osrm-extract -p /content/osrm-backend/profiles/car.lua /content/brazil-latest.osm.pbf

echo "🧩 Etapa B: Particionamento das Células (osrm-partition)..."
/content/osrm-backend/build/osrm-partition /content/brazil-latest.osrm

echo "🚦 Etapa C: Customização de Tráfego (osrm-customize)..."
/content/osrm-backend/build/osrm-customize /content/brazil-latest.osrm

echo "🎉 Todo o mapa rodoviário nacional foi processado matematicamente!"

⚙️ Etapa A: Extração do Grafo (osrm-extract)...
[2026-07-10T00:15:09.813167894] [info] Parsed 0 location-dependent features with 0 GeoJSON polygons
[2026-07-10T00:15:09.813295009] [info] Using script /content/osrm-backend/profiles/car.lua
[2026-07-10T00:15:09.813419473] [info] Input file: brazil-latest.osm.pbf
[2026-07-10T00:15:09.813434844] [info] Profile: car.lua
[2026-07-10T00:15:09.813437208] [info] Threads: 8
[2026-07-10T00:15:09.813755542] [info] Parsing in progress..
[2026-07-10T00:15:09.814123671] [info] input file generated by osmium/1.16.0
[2026-07-10T00:15:09.814302200] [info] timestamp: 2026-07-08T20:21:31Z
[2026-07-10T00:15:09.818177316] [info] Using profile api version 4
[2026-07-10T00:15:09.818580788] [info] Found 3 turn restriction tags:
[2026-07-10T00:15:09.818594488] [info]   motor_vehicle
[2026-07-10T00:15:09.818596943] [info]   motorcar
[2026-07-10T00:15:09.818598789] [info]   vehicle
[2026-07-10T00:15:09.818642754] [info] Parse relations ...
[2026-07-10T00:15:14.68

[2026-07-10T00:19:44.383059200] [warn] OSM node 13746251016 has a unidirectional traffic signal ambiguously applied to multiple ways
[2026-07-10T00:19:44.383132894] [warn] OSM node 256929997 has a unidirectional traffic signal ambiguously applied to multiple ways
[2026-07-10T00:19:44.383139104] [warn] OSM node 2068229852 has a unidirectional traffic signal ambiguously applied to multiple ways
[2026-07-10T00:19:44.383142415] [warn] OSM node 1972342215 has a unidirectional traffic signal ambiguously applied to multiple ways
[2026-07-10T00:19:44.383145420] [warn] OSM node 12759509114 has a unidirectional traffic signal ambiguously applied to multiple ways
[2026-07-10T00:19:44.383148483] [warn] OSM node 1100742949 has a unidirectional traffic signal ambiguously applied to multiple ways
[2026-07-10T00:19:44.383151567] [warn] OSM node 2514276555 has a unidirectional traffic signal ambiguously applied to multiple ways
[2026-07-10T00:19:44.383154693] [warn] OSM node 1404605541 has a unidirecti

In [ ]:
import os
import subprocess
import shutil

print("📦 Localizando e compactando os índices estruturados do OSRM...")
arquivos_osrm = [f for f in os.listdir('/content') if f.startswith('brazil-latest.osrm')]

if not arquivos_osrm:
    print("❌ Erro: Nenhum arquivo compilado encontrado na pasta /content.")
else:
    # Gera o arquivo .tar diretamente na raiz local rápida
    subprocess.run(["tar", "-cvf", "/content/brazil-latest.tar"] + [f"/content/{f}" for f in arquivos_osrm])

    origem = "/content/brazil-latest.tar"
    destino = "/content/drive/MyDrive/OSRM_Brasil_Compilado/brazil-latest.tar"

    print(f"💾 Enviando para o Google Drive por blocos seguros (shutil.copy2)...\")")
    try:
        shutil.copy2(origem, destino)
        if os.path.exists(destino):
            os.remove(origem) # Remove temporário local do Colab
            print("🎉 PROCESSO FINALIZADO SEM ERROS!")
            print(f"Seu mapa do Brasil inteiro pronto está seguro no seu Drive em: {destino}")
    except Exception as e:
        print(f"❌ Falha crítica ao transferir arquivo para o Drive: {e}")

📦 Localizando e compactando os índices estruturados do OSRM...
💾 Enviando para o Google Drive por blocos seguros (shutil.copy2)...")
🎉 PROCESSO FINALIZADO SEM ERROS!
Seu mapa do Brasil inteiro pronto está seguro no seu Drive em: /content/drive/MyDrive/OSRM_Brasil_Compilado/brazil-latest.tar
